# Import repaired BAFU EcoSpold files

Create or reuse a local Brightway project with **ecoinvent biosphere 3.10**, then try importing the repaired EcoSpold 1 files. Select the **bw** kernel and run the cells from top to bottom.

The repaired files must already exist. To generate them, run this from the repository root:

```bash
conda run --no-capture-output -n bw python "scripts/ecospold importer/repair_all.py"
```

Project storage is `artifacts/brightway/` (ignored by Git). The first project setup downloads Brightway's biosphere archive and requires internet access.

In [1]:
import os
import sys
from pathlib import Path


ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "scripts/ecospold importer").is_dir()
)
SOURCE = ROOT / "data/processed/ecospold1-schema-fixed"
STORAGE = ROOT / "artifacts/brightway"

PROJECT = "bafu-2026-biosphere-310"
BIOSPHERE = "ecoinvent-3.10-biosphere"
DATABASE = "BAFU:2026"

# Configure storage and the local helper path before importing Brightway.
STORAGE.mkdir(parents=True, exist_ok=True)
os.environ["BRIGHTWAY2_DIR"] = str(STORAGE)
sys.path.insert(0, str(ROOT / "scripts/ecospold importer"))

In [2]:
# Run the setup cell above first.
import bw2data as bd
import bw2io as bi
from date_compat import xml_date_parser
from timestamp_compat import iso_timestamp_parser

/opt/homebrew/Caskroom/miniforge/base/envs/bw/lib/python3.11/site-packages/scikits/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)


14:41:18+0200 [info     ] Using environment variable BRIGHTWAY2_DIR for data directory:
/Users/romain/GitHub/lca-data-lineage-hackathon/artifacts/brightway


## Create the project

Reuse the same project as the Python import script, or create it from Brightway's biosphere 3.10 archive on first use.

In [3]:
if PROJECT not in bd.projects:
    bi.install_project("ecoinvent-3.10-biosphere", project_name=PROJECT)
    
bd.projects.set_current(PROJECT)

In [4]:
bd.databases

Databases dictionary with 1 object(s):
	ecoinvent-3.10-biosphere

## Extract the repaired files

Use the local date/timestamp adapters and the standard EcoSpold 1 importer strategies. The XML files remain unchanged. The `importer` object stays available for inspection.

In [5]:
with iso_timestamp_parser(), xml_date_parser():
    importer = bi.SingleOutputEcospold1Importer(str(SOURCE), DATABASE, use_mp=False)

importer.apply_strategies()

  1%|▋                                                                            | 110/11947 [00:00<00:21, 546.93it/s]/opt/homebrew/Caskroom/miniforge/base/envs/bw/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:408: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
100%|███████████████████████████████████████████████████████████████████████████| 11947/11947 [00:24<00:00, 495.19it/s]


Extracted 11947 datasets in 24.20 seconds
Applying strategy: normalize_units
Applying strategy: assign_only_product_as_production
Applying strategy: clean_integer_codes
Applying strategy: drop_unspecified_subcategories
Applying strategy: strip_biosphere_exc_locations
Applying strategy: update_ecoinvent_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: link_technosphere_by_activity_hash
Applied 9 strategies in 1.36 seconds


## Apply the technosphere migrations

Edit the [general mappings](../schemas/mappings/bafu-2026-technosphere.json) or the [source-file-specific mappings](../schemas/mappings/bafu-2026-technosphere-context.json). The helper reads both files on every run, registers them in the current project, applies the corrections, and reruns technosphere linking.

The second file distinguishes identical exchange labels used by different consuming datasets. Its source-file marker is temporary and is removed after migration. See the [mapping notes](../schemas/mappings/README.md) for evidence and documented assumptions.

After changing either JSON file, rerun the extraction cell above and the following cells. Raw and repaired XML files remain unchanged.

In [6]:
from technosphere_migrations import apply_technosphere_migrations

apply_technosphere_migrations(importer, ROOT / "schemas/mappings")

Applying strategy: migrate_datasets
Applying strategy: migrate_exchanges
Applied 24 rules from bafu-2026-technosphere.json
Applying strategy: migrate_datasets
Applying strategy: migrate_exchanges
Applied 10 rules from bafu-2026-technosphere-context.json
Applying strategy: link_iterable_by_fields


In [7]:
importer.match_database(fields=["name", "reference product", "location"])

Applying strategy: link_iterable_by_fields


In [8]:
importer.match_database("ecoinvent-3.10-biosphere",fields=["name", "categories", "unit"])

Applying strategy: link_iterable_by_fields


In [9]:
importer.statistics()

Graph statistics for `BAFU:2026` importer:
11947 graph nodes:
	process: 11947
420063 graph edges:
	biosphere: 293747
	technosphere: 114369
	production: 11947
126316 edges to the following databases:
	BAFU:2026: 126316
2679 unique unlinked edges (293747 total):
	biosphere: 2679




(11947, 420063, 293747, 0)

In [ ]:
importer.data[0]

In [12]:
for u in list(importer.unlinked)[:10]:
    if u["type"] == "biosphere":
        print(u)

{'categories': ('emissions to air',), 'unit': 'megajoule', 'name': 'Heat, waste', 'type': 'biosphere', 'infrastructureProcess': False, 'comment': '(1,3,2,1,1,5); \n', 'uncertainty type': 2, 'amount': 0.366, 'loc': -1.0051219455807707, 'scale': 0.09942542937258259, 'negative': False}
{'categories': ('emissions to air', 'high. pop.'), 'unit': 'megajoule', 'name': 'Heat, waste', 'type': 'biosphere', 'infrastructureProcess': False, 'comment': '(3,5,5,1,3,5); Calculated from electricity use\n', 'uncertainty type': 2, 'amount': 0.0246, 'loc': -3.70500883604382, 'scale': 0.26236426446749106, 'negative': False}
{'categories': ('emissions to air', 'low. pop.'), 'unit': 'kilogram', 'name': 'Methane, fossil', 'type': 'biosphere', 'infrastructureProcess': False, 'comment': '(2,3,4,1,1,BU:1.5);Emissions from storage. Calculated based on average losses and gas composition ', 'uncertainty type': 2, 'amount': 1.8413e-06, 'loc': -13.205038714073236, 'scale': 0.22493234770242446, 'negative': False}
{'ca

In [ ]:
importer.drop_unlinked(i_am_reckless=True)

In [ ]:
importer.write_database()

In [ ]:
import bw2calc as bc
method = bd.methods.random()
act = bd.Database("BAFU:2026").random()
lca = bc.LCA({act: 1}, method)
lca.lci()
lca.lcia()
print(lca.score)

In [ ]:
act.as_dict()